Core thesis behind Loihi:
> Just as the value of ANNs was not fully appreciated until the advent of sufficiently fast CPUs and GPUs, the same could be the case for spiking models—except different computing architectures will be required.
- Loihi is a digital chip that is designed to work with synchronized neurons using a 'fixed-size discrete time-step model.'
- The chip assumes each neuron has two internal states: synaptic response current and membrane potential. A neuron only fires when one of them exceeds a certain threshold.
- The ℓ1‑minimizing sparse coding (LASSO) problem finds a sparse set of coefficients that reconstruct an input using a feature dictionary. In the spiking network, neurons compete so most coefficients stay near zero, and the average firing rates converge to a fixed point that matches the optimal sparse solution. As show below where the spiking rates between neurons stabilize (one spike cycle reaches a stable time):

![Neuron Potential across Cycles](neuron-potential.png)
- When compared to a convential optimizer (FISTA), a SNN solver (S-LCA) has a sub-accurate, yet fast solution to a sparse coding problem. Why?
	- SNNs utilize tempoeral ordering of spikes: meaning that neuron with largest input wins (and inhibit others) and fires/spikes at the earliest time. (See figure above)
	- Less overhead since one spike communicates/broadcasts the inhibition request to its competitor neurons, while classic solutions (i.e. FISTA) has more overhead (computationally, it multiples matrices, and communication-wise, it fetches past values to multiply them) because of 'all-to-all' state exchange.
- SNNs have a powerful feature: inherent parallism whereby individual neurons update their state independently. 
	- Loihi was designed to serve that since traditional CPUs would take time because of the accumulated overhead caused by its off-memory architecture
	- Loihi has no global clock, no kernel, and is an on-memory chip
- Loihi works well with SNNs because of how it has zero to minimal overhead (nanoseconds) and is mostly negligible. So instead of $T_{p} = N / p * t_{u} + t_{overhead}$ (time to complete one timestamp with P workers), it's $T_{p}\propto N / p * t_{u}$
- As a (digital) SNN chip, Loihi has more flexibility and more network capacity than other neuromorphic chips:
	- Spike tracing in programmable manner (setting or filtering time constants)
	- Additional state variables in addition to membrane potential and synaptic current
	- Support for RL via reward spikes (carrying signed values to reward or punish for RL)

Learning with locality

- Any SNN must satisfy the locality constraint: each weight is updated based only on its source and destination neurons, and used and changed only by its desintation neuron
	- Efficient large-scale computation would then be acheieved via adherence to locality constraint using learning rules, e.g. Oja's rule (1st PCA), Widrow-Hoff Rule, event-driven random backprop. 
	- And if a loss function is minimized via the learning rule, then well-defined dynamics (interpretable results) are achieved.
- Loihi also offers advanced SNN capabilities:
	- Random noise to neurons' signals & time for more exploration
	- Let spikes arrive late so neurons line up in time
	- Splitting neurons for richer input processing
	- Neurons with homeostasis (stable firing, not too active or too silent neurons)
	- Weights are grown beyond normal inference range but still capped (i.e. long-term memory of connectivity)

Chip architecture

- The Loihi chip has 128 neuromorphic cores (for spike messages), 3 embedded x86 processor cores, and off-chip comm interfaces
	> The NoC supports write, read re-quest, and read response messages for core management and x86-to-x86 messaging, spike messages for SNN computation, and barrier messages for time synchronization between cores. ... The mesh protocol supports scaling to 4096 on-chip cores and, through hierarchical addressing, up to 16,384 chips.
- Algorithmic performance of SNNs improved as we add 1) more neurons and 2) giving each neuron more outgoing connections. Yet this incremental improvement breeds exponential complexity O(n^2) in the number of fan-outs that cannot be served by today's IC technology
- To solve this, Loihi relaxes constraints other chips imposed:
	- Storing compact version of network with synapses carrying index state
	- One spike to many destination cores, efficient messaging
	- Weights can 1-9 bits, signed or signed, for neuron's connections providing precision when needed
	- Weight sharing via connectivity template

Mesh Operation
- In a mesh operation, cores starting at time `t` (initially in idle state) would need to send their neuron's spiking messages (once it fires) that NoC distributes to synaptic fan-outs (target cores) using dimension order routing algorithm
- While it doesn't actually do multicasting of these messages, it iterates the list of target neurons one spike per core
- To avoid deadlock, mesh uses two independent physical router networks for messaging. And to achieve bandwidth efficiency (i.e. data capacity is at bay), cores alternate sending spikes between the two networks
- Before moving to `t+1`, local handshakes (rather than global clock) are used to ensure all spiking messages are sent
- Between `t` and `t+1`, management traffic (read/write config registers, C2C communication, debugging, all for reliability and monitoring purposes) can happen without resulting in deadlock

Network Connectivity Architecture
- The neural network mapped onto the Loihi architecture is a directed multipgraph structure `g = (N, S)` where each synapse has (i, j, wgt, dly, tag) where i and j are source and desintation neurons. Others are integer-valued properties. And wgt as synapse strength, dly as delay, and tag as metadata required by hardware.
- In that network, neurons are mapped onto cores and then connected to neurons via synapses. When neuron connect to other neurons, they go with axon_id label which has destination core, source neuron index, and template edge (abstract connection rule in hierarchial model of the neural network)
- These networks have constraints, namely: total number of neurons per core is 1024, total synaptic fan-in state mapped to a core is 128 KB, total number of core-to-core fan-out edges mapped to any core <= 4096, total number of fan-out is 4096
- Hierarchical model of the network saves memory and routing resources but creates more spike traffic
	- CNNs are NNs that greatily benefit from this model + inhibitory connections as in the S-LCA network

Learning Engine
- The loihi chip, rather than than calculating synaptic weight immediately after each spike event, it accumulates pre/post-synaptic traces and then after each epoch ends, it sequentially applies learning updates to the relevant synapses using those traces
	- For STDP rule:
		- if presynaptic spike fires before postsynaptic spike, then the traces gives a positive update
		- if postsynaptc spike fires before presynaptic spike, then the traces gives a negative update
		- conceptually, new weight = old weight + contribution from recent pre/post timing
- This mechanism gives flexibility to the usage of these traces, and thus flexibility to use other learning rules.
	- an example if reward-modulated learning: new weight = old weight + contribution from eligibility trace × reward signal
		- where eligibility trace is whether the synapse recently participate in activity (positive if yes, negative if not)

![Functional Learning Rule](functional-learning-rule.png)
- The equation show that with each learning epoch, synaptic vars (weight, delay, tag) are updated from a combo of (local) pre/post traces, reward traces, and current synaptic vars (as well as constants)
- where conceptually the new value of the synaptic var = sum of several terms and each term = constant * product of selected local vars mention above 

Microarchitecture
- Each core of the Loihi chip is a neuromorphic engine with memory that processes synapses, updates neurons, outputs spikes and learns rules.
- RMW memory accesses are implemented around SYNAPSE_MEM, POST_TRACE, DENDRITE_ACCCUM, SYNAPSE_MAP, PRE_TRACE, CX META_STATE, others
- "The SYNAPSE unit processes all incoming spikes and reads out the associated synaptic weights from the memory. The DENDRITE unit updates the state variables u and v of all neurons in the core. The AXON unit generates spike messages for all fan-out cores of each firing neuron. The LEARNING unit updates synaptic weights using the programmed learning rules at epoch boundaries."

Async Design Methodology
- The chip's design is improved on an earlier async design methodology used to develop commercial Ethernet switches

Results
- Device is functional over a supply voltage range of 0.50-1.25 V
- Includes 16 MB of synaptic memory with 2.1M unique synaptic variables per mm^2 = 3x higher than TrueNorth
- 18x compression in synaptic resources when solving for CSCP